# Baseline B1 - evaluation

Author: Mathias Marryat Husted

This notebook provides a reproducible way to evaluate the baseline run B1. Checks are performed at each step to ensure data consistency.

In [72]:
import pandas as pd
import math

In [73]:
BENCHMARK_PATH = "../../data/data set/benchmark.csv"

benchmark = pd.read_csv(BENCHMARK_PATH, dtype=str, keep_default_na=False)

### Load the data

The data from B1 is divided into 3 groups:
- B1A/B1B (byte-exact vs case-folded)
- Genes/chemicals
- Canonical/synomym forms

The matches have been established in the previous step and the resulting files have been created. The goal is to extract useful information from this data.

The file system corresponds to the 4x2 grid resulting from the division (8 files in total):

| | Chemicals (canonical) | Chemicals (synonyms) | Genes (canonical) | Genes (synonyms) |
|--|--|--|--|--|
| B1a (exact byte-by-byte) | _b1a_chem_canonical.csv_ | _b1a_chem_synonyms.csv_ | _b1a_gene_canonical.csv_ | _b1a_gene_synonym.csv_ |
| B1b (case-folded) | _b1b_chem_canonical.csv_ | _b1b_chem_synonym.csv_ | _b1b_gene_canonical.csv_ | _b1b_gene_synonym.csv_ |

Each key has been matched to the database according to the rules that follows from its file name.

In [74]:
B1_PATH = "../results"
suffixes = ["a", "b"]
types = ["chem", "gene"]
tiers = ["canonical", "synonym"]
b1_data = [
    {
        'suffix': suffix,
        'type': type_name,
        'tier': tier,
        'df': pd.read_csv(f"{B1_PATH}/b1{suffix}_{type_name}_{tier}.csv", dtype=str, keep_default_na=False)
    }
    for suffix in suffixes
    for type_name in types
    for tier in tiers
]

# First, we assert the data has the correct expected row numbers
genes = [item for item in b1_data if item['df'].shape[0] == 157 and item['type'] == "gene"]
chemicals = [item for item in b1_data if item['df'].shape[0] == 162 and item['type'] == "chem"]
assert len(genes) == 4 and len (chemicals) == 4, "The files do not have the expected row count. Data might be corrupted"

# Next, we check all items have a corresponding entry in the benchmark
benchmark_gene_ids = set(benchmark[benchmark["entity_type"] == "gene"]["id"])
benchmark_chemical_ids = set(benchmark[benchmark["entity_type"] == "chemical"]["id"])

for gene in genes:
    assert set(gene['df']['id']) == benchmark_gene_ids, "Not all gene IDs match the benchmark"

for chemical in chemicals:
    assert set(chemical['df']['id']) == benchmark_chemical_ids, "Not all chemical IDs match the benchmark"

# Now, we check check the expected columns are present

for item in b1_data:
    required_columns = {"id", "input", "n_candidates", "returned_id"}
    if item['type'] == "gene":
        required_columns.add("matched_field")
    missing = required_columns - set(item['df'].columns)
    assert not missing, f"b1{item['suffix']}_{item['type']}_{item['tier']}: missing columns {missing}"

# Fourth, we join on the IDs and check that no inputs have changed
gold_columns = ["id", "input", "canonical_name", "entity_type", "noise_type", "id_status", "accepted_id"]

for item in b1_data:
    prediction_columns = ["id", "input", "n_candidates", "returned_id"]
    if item['type'] == "gene":
        prediction_columns.append("matched_field")

    item['merged'] = pd.merge(
        left=item['df'][prediction_columns],
        right=benchmark[gold_columns],
        on="id",
        how="left",
        suffixes=("_result", "_benchmark"),
        validate="one_to_one"
    )
    assert (item['merged']['input_result'] == item['merged']['input_benchmark']).all(), f"b1{item['suffix']}_{item['type']}_{item['tier']}: inputs different from benchmark.csv, result file is stale"


# Fifth, we check our policy is implemented correctly; n_candidates converts to int, is >=0 and returned_id is not empty iff n_candidates == 1

for item in b1_data:
    item['merged']['n_candidates'] = item['merged']['n_candidates'].astype(int)
    for row in item['merged'].itertuples():
        assert row.n_candidates >= 0, f"Negative n_candidates on row {row.id}"
        if row.n_candidates == 1:
            assert row.returned_id != "", f"One candidate but no returned id on row {row.id}"
        else:
            assert row.returned_id == "", f"returned_id despite not exactly one candidate on row {row.id}"

In [75]:
# Decision function
# Each item  in the table will get one of six possible outcomes:
# returned_on_none: id_status was none, but returned_id still wasn't empty
# abstained_on_none: id_status was none and returned_id correctly came back empty
# abstain_no_match: id_status wasn't none, but no matches were found
# abstain_ambig: id_status wasn't none, there was more than one candidate
# correct: id_status wasn't none, there was exactly one condidate and it was in the accepted_id list
# wrong_entity: if all else fails, the classification was wrong to begin with
def decide(row):
    if row['id_status'] == "none":
        if row['returned_id'] != "":
            return "returned_on_none"
        return "abstained_on_none"
    
    if row['n_candidates'] == 0:
        return "abstain_no_match"
    
    if row['n_candidates'] > 1:
        return "abstain_multiple"
    if row['returned_id'] in row['accepted_id'].split(";"):
        return "correct"
    return "wrong_entity"

In [76]:
# Create a new data frame to store all of the 8 tables concatenated into one
dfs_b2 = []
for item in b1_data:
    df = item['merged'].copy()
    df['case'] = item['suffix']
    df['tier'] = item['tier']
    df['outcome'] = df.apply(decide, axis=1)
    dfs_b2.append(df)

b1_full = pd.concat(dfs_b2, ignore_index=True)
assert len(b1_full) == 4 * 157 + 4 * 162

In [77]:
# Check
pd.crosstab([b1_full['entity_type'], b1_full['case'], b1_full['tier']], b1_full['outcome'])

outcome                     abstain_multiple  abstain_no_match  \
entity_type case tier                                            
chemical    a    canonical                 0               149   
                 synonym                   2               107   
            b    canonical                 0               130   
                 synonym                   2                57   
gene        a    canonical                 0               149   
                 synonym                   2                99   
            b    canonical                 0               129   
                 synonym                   3                77   

outcome                     abstained_on_none  correct  returned_on_none  \
entity_type case tier                                                      
chemical    a    canonical                 10        3                 0   
                 synonym                    9       38                 1   
            b    canonical                 10       17                 0   
                 synonym                    9       83                 1   
gene        a    canonical                  8        0                 0   
                 synonym                    8       43                 0   
            b    canonical                  8       20                 0   
                 synonym                    8       64                 0   

outcome                     wrong_entity  
entity_type case tier                     
chemical    a    canonical             0  
                 synonym               5  
            b    canonical             5  
                 synonym              10  
gene        a    canonical             0  
                 synonym               5  
            b    canonical             0  
                 synonym               5

In [78]:
# Number of successful resolutions
# Quote: Wilson (1927), the original: E. B. Wilson, "Probable inference, the law of succession, and statistical inference," Journal of the American Statistical Association 22(158), 209–212.

def wilson_score_interval(ups, total, z=1.96): # z=1.96 -> Calculate the 95% confidence interval
    if total == 0:
        return 0

    p = ups / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    half_w = z * math.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denominator
    return (max(0.0, center - half_w), min(1.0, center + half_w))


## Headline table

In [79]:
# Accuracy = correct / n
# Coverage = (correct + wrong) / n
# Precision = correct / (correct + wrong)
id_status_single = b1_full[b1_full['id_status'] == "single"]

grouped = id_status_single.groupby(['entity_type', 'case', 'tier'])

headline_table = {
    "Entity type": [key[0] for key, group in grouped],
    "Case": [("exact" if key[1] == "a" else "folded") for key, group in grouped],
    "Tier": [key[2] for key, group in grouped],
    "n": [len(group) for keys, group in grouped],
    "Correct": [(group['outcome'] == "correct").sum() for key, group in grouped],
    "Accuracy": [round((group['outcome'] == "correct").sum()/len(group), 3) for key, group in grouped],
    "95% CI": [tuple(round(x, 3) for x in wilson_score_interval((group['outcome'] == "correct").sum(), len(group))) for key, group in grouped],
    "Coverage": [round(group['outcome'].isin(["correct", "wrong_entity"]).sum() / len(group), 3) for key, group in grouped],
    "Precision": [round((group['outcome'] == "correct").sum() / group['outcome'].isin(["correct", "wrong_entity"]).sum(), 3) for key, group in grouped],
    "Wrong entity": [(group['outcome'] == "wrong_entity").sum() for key, group in grouped]
    }

headline_table = pd.DataFrame(data=headline_table)
headline_table

/tmp/ipykernel_925/1046237443.py:17: RuntimeWarning: invalid value encountered in scalar divide
  "Precision": [round((group['outcome'] == "correct").sum() / group['outcome'].isin(["correct", "wrong_entity"]).sum(), 3) for key, group in grouped],


,Entity type,Case,Tier,n,Correct,Accuracy,95% CI,Coverage,Precision,Wrong entity
0,chemical,exact,canonical,106,3,0.028,"(0.01, 0.08)",0.028,1.0,0
1,chemical,exact,synonym,106,30,0.283,"(0.206, 0.375)",0.283,1.0,0
2,chemical,folded,canonical,106,14,0.132,"(0.08, 0.21)",0.132,1.0,0
3,chemical,folded,synonym,106,66,0.623,"(0.528, 0.709)",0.623,1.0,0
4,gene,exact,canonical,119,0,0.000,"(0.0, 0.031)",0.000,NaN,0
5,gene,exact,synonym,119,41,0.345,"(0.265, 0.434)",0.345,1.0,0
6,gene,folded,canonical,119,11,0.092,"(0.052, 0.158)",0.092,1.0,0
7,gene,folded,synonym,119,53,0.445,"(0.359, 0.535)",0.445,1.0,0


## Table grouped by noise type

In [80]:
# Filter synonym and single rows
synonym_data = b1_full[(b1_full['id_status'] == "single") & (b1_full['tier'] == "synonym")]

grouped = synonym_data.groupby(['entity_type', 'noise_type'])

def calc_accuracy_ci(group, case):
    filtered = group[group['case'] == case]
    correct_count = (filtered['outcome'] == "correct").sum()
    low, high = wilson_score_interval(correct_count, len(filtered))
    return f"{correct_count}/{len(filtered)} {100 * correct_count / len(filtered):.1f} [{100 * low:.1f}–{100 * high:.1f}]"

def calc_accuracy(group, case):
    filtered = group[group['case'] == case]
    return (filtered['outcome'] == "correct").sum() / len(filtered)

# Build the table data
table_rows = []
for key, group in grouped:
    assert len(group[group['case'] == "a"]) == len(group[group['case'] == "b"])
    entity_type, noise_type = key
    exact = calc_accuracy_ci(group, "a")
    folded = calc_accuracy_ci(group, "b")
    gain = round(100 * (calc_accuracy(group, "b") - calc_accuracy(group, "a")), 1)
    table_rows.append([entity_type, noise_type, exact, folded, gain])

class_table = pd.DataFrame(table_rows, columns=["Entity type", "Noise type", "Exact", "Folded", "Fold gain (pp)"])

class_table


,Entity type,Noise type,Exact,Folded,Fold gain (pp)
0,chemical,abbreviation,10/12 83.3 [55.2–95.3],11/12 91.7 [64.6–98.5],8.3
1,chemical,casing,1/13 7.7 [1.4–33.3],13/13 100.0 [77.2–100.0],92.3
2,chemical,separator,1/12 8.3 [1.5–35.4],9/12 75.0 [46.8–91.1],66.7
3,chemical,synonym,13/24 54.2 [35.1–72.1],21/24 87.5 [69.0–95.7],33.3
4,chemical,tool_artifact,0/10 0.0 [0.0–27.8],0/10 0.0 [0.0–27.8],0.0
5,chemical,typo,0/18 0.0 [0.0–17.6],1/18 5.6 [1.0–25.8],5.6
6,chemical,unperturbed,5/7 71.4 [35.9–91.8],7/7 100.0 [64.6–100.0],28.6
7,chemical,word_order,0/10 0.0 [0.0–27.8],4/10 40.0 [16.8–68.7],40.0
8,gene,abbreviation,20/21 95.2 [77.3–99.2],20/21 95.2 [77.3–99.2],0.0
9,gene,casing,0/11 0.0 [0.0–25.9],11/11 100.0 [74.1–100.0],100.0


In [81]:
wrong_rows = b1_full[b1_full['outcome'].isin(["wrong_entity", "returned_on_none"])]
wrong_rows = wrong_rows[["id", "case", "tier", "input_benchmark", "entity_type", "noise_type", "id_status",
                        "matched_field", "returned_id", "accepted_id", "outcome"]]
wrong_rows = wrong_rows.sort_values(["id", "tier", "case"])
#wrong_rows.to_csv(f"{B1_PATH}/wrong_rows/wrong_rows.csv")
wrong_rows

,id,case,tier,input_benchmark,entity_type,noise_type,id_status,matched_field,returned_id,accepted_id,outcome
827,CHEM-028,b,synonym,DHA,chemical,abbreviation,ambiguous,NaN,15608515,445580,wrong_entity
190,CHEM-029,a,synonym,EPA,chemical,abbreviation,ambiguous,NaN,446284,5282847,wrong_entity
828,CHEM-029,b,synonym,EPA,chemical,abbreviation,ambiguous,NaN,446284,5282847,wrong_entity
192,CHEM-031,a,synonym,G6P,chemical,abbreviation,ambiguous,NaN,439284,439958;99058,wrong_entity
830,CHEM-031,b,synonym,G6P,chemical,abbreviation,ambiguous,NaN,439284,439958;99058,wrong_entity
201,CHEM-040,a,synonym,PBS,chemical,abbreviation,ambiguous,NaN,14819,24978514,wrong_entity
839,CHEM-040,b,synonym,PBS,chemical,abbreviation,ambiguous,NaN,14819,24978514,wrong_entity
697,CHEM-060,b,canonical,Alpha-ketoglutarate,chemical,casing,ambiguous,NaN,164533,51,wrong_entity
859,CHEM-060,b,synonym,Alpha-ketoglutarate,chemical,casing,ambiguous,NaN,164533,51,wrong_entity
722,CHEM-085,b,canonical,vitamin B6,chemical,synonym,ambiguous,NaN,104817,1054,wrong_entity


This resolution of the wrong rows helped inform the corrections to the data set. The wrong rows remaining in the list have all been corrected or clarified to be consistent with the rule set.

In [82]:
ambig_rows = b1_full[b1_full['id_status'] == "ambiguous"]
pd.crosstab([ambig_rows['entity_type'], ambig_rows['case'], ambig_rows['tier']],
            ambig_rows['outcome'])

outcome                     abstain_multiple  abstain_no_match  correct  \
entity_type case tier                                                     
chemical    a    canonical                 0                46        0   
                 synonym                   2                31        8   
            b    canonical                 0                38        3   
                 synonym                   2                17       17   
gene        a    canonical                 0                30        0   
                 synonym                   1                22        2   
            b    canonical                 0                21        9   
                 synonym                   1                13       11   

outcome                     wrong_entity  
entity_type case tier                     
chemical    a    canonical             0  
                 synonym               5  
            b    canonical             5  
                 synonym              10  
gene        a    canonical             0  
                 synonym               5  
            b    canonical             0  
                 synonym               5

Major findings about baseline B1:
- Accuracy is generally low. Its high precision comes from low coverage. Exact string matching isn't a reliable normalization technique
- Exact matching is precise but brittle. All answers on "single" rows are correct, its failure come mainly from abstentions rather than errors
- Precision remained at 100% throughout. Case-folding did **not** cost precision
- Case-folding helped chemicals far more than genes
- The biggest improvement came from including synonyms in the vocabulary. The step from the canonical -> synonym tiers was the biggest improvement for both chemicals and genes
- Even though the baseline performed exact matching and fabrication was impossible, wrong entities can still be returned
    - KARS/KRAS: The typo itself matched a different gene and therefore got wrongfully matched
    - Competing referents (alias/prev_symbol): An entity may have been matched to an alias or a previous symbol (e.g. COX-2 -> PTGER2)
    - Formula parsing: Pubchem in some cases parsed a wrong name as a chemical formula (e.g. N -> nitrogen, PbS -> lead sulfide)
    - Related compounds: Underspecified inputs lead to matching with sibling records (e.g. Lactate -> stereo-unspecified lactate, G6P -> $\alpha$-G6P)

# B2 evaluation

In [83]:
benchmark_chemicals = benchmark[benchmark["entity_type"] == "chemical"]
benchmark_genes = benchmark[benchmark["entity_type"] == "gene"]

b2_chem = pd.read_csv("../results/b2_chem_synonym.csv", dtype=str, keep_default_na=False)
b2_gene = pd.read_csv("../results/b2_gene_synonym.csv", dtype=str, keep_default_na=False)

b2_data = [
    {"type": "chem", "df": b2_chem, "bench": benchmark_chemicals,},
    {"type": "gene", "df": b2_gene, "bench": benchmark_genes,},
]

In [84]:
# First, we run similar data integrity checks as in B1
assert len(b2_chem) == 162 and len(b2_gene) == 157


for item in b2_data:

    required_columns = {"id", "input", "n_candidates", "returned_id"}
    if item["type"] == "gene":
        required_columns.add("matched_field")
    missing = required_columns - set(item["df"].columns)
    assert not missing, f"b2_{item["type"]}: missing columns {missing}"

# Now, we join our tables into one
gold_columns = ["id", "input", "canonical_name", "entity_type", "noise_type", "id_status", "accepted_id"]

for item in b2_data:
    prediction_columns = ["id", "input", "n_candidates", "returned_id", "score", "candidate_ids", "matched_string"]
    if item['type'] == "gene":
        prediction_columns.append("matched_field")

    item['merged'] = pd.merge(
        left=item['df'][prediction_columns],
        right=benchmark[gold_columns],
        on="id",
        how="left",
        suffixes=("_result", "_benchmark"),
        validate="one_to_one"
    )
    assert (item['merged']['input_result'] == item['merged']['input_benchmark']).all(), f"b2{item["type"]}: inputs different from benchmark.csv, result file is stale"

In [85]:
# With our merged data frame, we can now apply the same decision function on B2 as on B1

dfs_b2 = []
for item in b2_data:
    df = item['merged'].copy()
    df['n_candidates'] = df['n_candidates'].astype(int)
    df['score'] = df['score'].astype(float)
    df['outcome'] = df.apply(decide, axis=1)
    df['type'] = item['type']
    df['method'] = "B2"
    dfs_b2.append(df)

b2_full = pd.concat(dfs_b2, ignore_index=True)
b2_full_frame = pd.DataFrame(b2_full)
b2_full_frame.to_csv("b2_outcomes.csv", index=False)

In [86]:
pd.crosstab(b2_full["type"], b2_full['outcome'])

outcome,abstain_multiple,abstained_on_none,correct,returned_on_none,wrong_entity
type,,,,,
chem,12,3,114,7,26
gene,28,5,93,3,28


In [87]:
# Interesting insight: Wrong entities where the score was 100
#b2_full_frame[(b2_full_frame["score"] == 100.0 )& (b2_full_frame["outcome"] != "correct")]
#b2_full_frame[b2_full_frame["outcome"] != "correct"]

b2_grouped = b2_full[b2_full['id_status'] == "single"].groupby('type')


b2_headline = {
    "Entity type": [key for key, group in b2_grouped],
    "Method": ["B2 (folded, fuzzy, synonym)" for key, group in b2_grouped],
    "n": [len(group) for keys, group in b2_grouped],
    "Correct": [(group['outcome'] == "correct").sum() for key, group in b2_grouped],
    "Accuracy": [round((group['outcome'] == "correct").sum() / len(group), 3) for key, group in b2_grouped],
    "95% CI": [tuple(round(x, 3) for x in wilson_score_interval((group['outcome'] == "correct").sum(), len(group))) for key, group in b2_grouped],
    "Coverage": [round(group['outcome'].isin(["correct", "wrong_entity"]).sum() / len(group), 3) for key, group in b2_grouped],
    "Precision": [round((group['outcome'] == "correct").sum() / group['outcome'].isin(["correct", "wrong_entity"]).sum(), 3) for key, group in b2_grouped],
    "Wrong entity": [(group['outcome'] == "wrong_entity").sum() for key, group in b2_grouped],
}
b2_headline = pd.DataFrame(data=b2_headline)
b2_headline

,Entity type,Method,n,Correct,Accuracy,95% CI,Coverage,Precision,Wrong entity
0,chem,"B2 (folded, fuzzy, synonym)",106,91,0.858,"(0.78, 0.912)",0.962,0.892,11
1,gene,"B2 (folded, fuzzy, synonym)",119,81,0.681,"(0.592, 0.758)",0.832,0.818,18


In [88]:
# We can immediately see an increase in accuracy over both types
# Best B1 result for chemicals: folded + synonyms
#3	chemical	folded	synonym	106	66	0.623	(0.528, 0.709)	0.623	1.0	0
increase_acc_chem = b2_headline[b2_headline["Entity type"] == "chem"]["Accuracy"] - 0.623
# Best B1 result for genes: folded + synonyms
#7	gene	folded	synonym	119	53	0.445	(0.359, 0.535)	0.445	1.0	0
increase_acc_gene = b2_headline[b2_headline["Entity type"] == "gene"]["Accuracy"] - 0.445
decrease_prec_chem = 1 - b2_headline[b2_headline["Entity type"] == "chem"]["Precision"]
decrease_prec_gene = 1 - b2_headline[b2_headline["Entity type"] == "gene"]["Precision"]

print(f"Increase in accuracy: {increase_acc_chem}/{increase_acc_gene}, decrease in precision: {decrease_prec_chem}/{decrease_prec_gene}")

Increase in accuracy: 0    0.235
Name: Accuracy, dtype: float64/1    0.236
Name: Accuracy, dtype: float64, decrease in precision: 0    0.108
Name: Precision, dtype: float64/1    0.182
Name: Precision, dtype: float64


### Findings:
- Significant increase in accuracy (About 24% over the best B1 run, for both types)
- Precision went down with accuracy (10.8/18.2% respectively)

In [89]:
ambig_rows = b2_full[b2_full['id_status'] == "ambiguous"]
pd.crosstab([ambig_rows['type']], ambig_rows['outcome'])

outcome,abstain_multiple,correct,wrong_entity
type,,,
chem,8,23,15
gene,8,12,10


Let us now compare the results grouped by the entity and noise types. We will compare directly to B1b, as that measures exactly the difference of introducing fuzzy matching.

In [91]:
# Quick helper function
def class_accuracy(group):
    return (group['outcome'] == "correct").sum() / len(group)

# Filter single rows
b1b_single = b1_full[(b1_full['id_status'] == "single") & (b1_full['case'] == "b") & (b1_full['tier'] == "synonym")]
b2_single = b2_full[b2_full['id_status'] == "single"]
b2_grouped = b2_single.groupby(['entity_type', 'noise_type'])


# Build the table data
b2_table_rows = []
for key, group in b2_grouped:
    entity_type, noise_type = key

    correct_count = (group['outcome'] == "correct").sum()
    low, high = low, high = wilson_score_interval(correct_count, len(group))
    b2_acc = f"{correct_count}/{len(group)} {100 * correct_count / len(group):.1f} [{100 * low:.1f}–{100 * high:.1f}]"
    answered = group['outcome'].isin(["correct", "wrong_entity"]).sum()
    wrong = (group['outcome'] == "wrong_entity").sum()

    b1b_group = b1b_single[(b1b_single['entity_type'] == entity_type) & (b1b_single['noise_type'] == noise_type)]
    b1b_acc = f"{(b1b_group['outcome'] == "correct").sum()}/{len(b1b_group)}"
    group_diff = round(100 * (class_accuracy(group) - class_accuracy(b1b_group)), 1)

    b2_table_rows.append([entity_type, noise_type, b1b_acc, b2_acc, group_diff, answered, wrong])


b2_class_table = pd.DataFrame(b2_table_rows, columns=["Entity type", "Noise type", "B1b accuracy", "B2 accuracy", "Class gain over B1b (pp)", "Answered", "Wrong entity"])
b2_class_table

,Entity type,Noise type,B1b accuracy,B2 accuracy,Class gain over B1b (pp),Answered,Wrong entity
0,chemical,abbreviation,11/12,11/12 91.7 [64.6–98.5],0.0,12,1
1,chemical,casing,13/13,13/13 100.0 [77.2–100.0],0.0,13,0
2,chemical,separator,9/12,12/12 100.0 [75.7–100.0],25.0,12,0
3,chemical,synonym,21/24,21/24 87.5 [69.0–95.7],0.0,22,1
4,chemical,tool_artifact,0/10,6/10 60.0 [31.3–83.2],60.0,9,3
5,chemical,typo,1/18,16/18 88.9 [67.2–96.9],83.3,17,1
6,chemical,unperturbed,7/7,7/7 100.0 [64.6–100.0],0.0,7,0
7,chemical,word_order,4/10,5/10 50.0 [23.7–76.3],10.0,10,5
8,gene,abbreviation,20/21,20/21 95.2 [77.3–99.2],0.0,20,0
9,gene,casing,11/11,11/11 100.0 [74.1–100.0],0.0,11,0


Grouping by noise type, we can draw the following conclusions about fuzzy matching:
- It provides a significant improvement to the string matching on 8/15 classes
- It does not give a loss on the classes where B1b was already at 100%
- It rescues the classes that exact matching could not handle, particularly `typo`, `tool_artifact`, and `separator`. `word_order` also saw a boost, especially on genes
- This happens by the fact it always answers. Therefore, accuracy must viewed read next to precision.
- On genes, `synonym` saw a slight decrease due to a single previously matched gene now now longer being mapped. This could be the cost of increasing uncertainty 
- Fuzzy matching seems to be a serious contender to an LLM-based solution
- To beat fuzzy matching, an LLM would have to be further improve on the types which fuzzy matching is already more successful at